## Классические алгоритмы без ансамблирования
В этом ноутбуке вам нужно обучить модели на датасете классификации из предыдущего ноутбука и сравнить результаты. Вам будет предоставлен baseline, на основе которого вы будете доделывать предсказывающие модели. Оценка лабы будет зависеть от ROC-AUC на тестовых данных по следующим критериям:
\
AUC - на тестовых данных
- $AUC \leq 0.75$ - 0 баллов
- $0.75 < AUC \leq 0.76$ - 2 балла
- $0.76 < AUC \leq 0.77$ - 4 балла
- $0.77 < AUC \leq 0.78$ - 6 баллов
- $0.78 < AUC \leq 0.79$ - 8 баллов
- $AUC > 0.79$ - 10 баллов

\
В этой работе запрещено использовать ансамбли моделей (лес, бустинги и т.д.)!

In [ ]:
%matplotlib inline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score, precision_score, recall_score, roc_curve, accuracy_score

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('german.csv', sep=';')
print(data.head())

X = data.iloc[:, 1:].to_numpy()
y = data.iloc[:, 0].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
plt.hist(y_train, bins=2, edgecolor='k')
plt.xticks([0, 1])
plt.xlabel('Class (0: Non-Creditworthy, 1: Creditworthy)')
plt.ylabel('Count')
plt.title('Distribution of Classes in Training Data')
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Создание модели Logistic Regression
logistic_regression_model = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
logistic_regression_model.fit(X_train, y_train)

# Создание модели Decision Tree
decision_tree_model = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
decision_tree_model.fit(X_train, y_train)

# Создание модели K-Nearest Neighbors
knn_model = KNeighborsClassifier(n_neighbors=15, metric='euclidean')
knn_model.fit(X_train, y_train)

In [ ]:
y_prob_logistic = logistic_regression_model.predict_proba(X_test)[:, 1]
y_prob_decision_tree = decision_tree_model.predict_proba(X_test)[:, 1]
y_prob_knn = knn_model.predict_proba(X_test)[:, 1]

y_pred_logistic = logistic_regression_model.predict(X_test)
y_pred_decision_tree = decision_tree_model.predict(X_test)
y_pred_knn = knn_model.predict(X_test)

accuracy_logistic = accuracy_score(y_test, y_pred_logistic)
accuracy_decision_tree = accuracy_score(y_test, y_pred_decision_tree)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

roc_auc_logistic = roc_auc_score(y_test, y_prob_logistic)
roc_auc_decision_tree = roc_auc_score(y_test, y_prob_decision_tree)
roc_auc_knn = roc_auc_score(y_test, y_prob_knn)

precision_logistic = precision_score(y_test, y_pred_logistic)
precision_decision_tree = precision_score(y_test, y_pred_decision_tree)
precision_knn = precision_score(y_test, y_pred_knn)

recall_logistic = recall_score(y_test, y_pred_logistic)
recall_decision_tree = recall_score(y_test, y_pred_decision_tree)
recall_knn = recall_score(y_test, y_pred_knn)

print(f'Accuracy of Logistic Regression: {accuracy_logistic}')
print(f'Accuracy of Decision Tree: {accuracy_decision_tree}')
print(f'Accuracy of K-Nearest Neighbors: {accuracy_knn}')

print(f'ROC AUC of Logistic Regression: {roc_auc_logistic}')
print(f'ROC AUC of Decision Tree: {roc_auc_decision_tree}')
print(f'ROC AUC of K-Nearest Neighbors: {roc_auc_knn}')

print(f'Precision of Logistic Regression: {precision_logistic}')
print(f'Precision of Decision Tree: {precision_decision_tree}')
print(f'Precision of K-Nearest Neighbors: {precision_knn}')

print(f'Recall of Logistic Regression: {recall_logistic}')
print(f'Recall of Decision Tree: {recall_decision_tree}')
print(f'Recall of K-Nearest Neighbors: {recall_knn}')

## Экспериментируйте
Для получения лучшего качества придется поэкспериментировать. Подсказка: попробуйте оптимизировать гиперпараметры модели

In [ ]:
from sklearn.model_selection import GridSearchCV

# Подбор гиперпараметров для Logistic Regression
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    {'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]},
    scoring='roc_auc', cv=5, n_jobs=-1
)
lr_grid.fit(X_train, y_train)
print('Лучшие параметры LR:', lr_grid.best_params_, '| CV AUC:', round(lr_grid.best_score_, 4))

# Подбор гиперпараметров для Decision Tree
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    {'max_depth': [3, 4, 5, 6, 7], 'min_samples_leaf': [5, 10, 15, 20]},
    scoring='roc_auc', cv=5, n_jobs=-1
)
dt_grid.fit(X_train, y_train)
print('Лучшие параметры DT:', dt_grid.best_params_, '| CV AUC:', round(dt_grid.best_score_, 4))

# Подбор гиперпараметров для KNN
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': [5, 10, 15, 20, 25, 30, 40], 'metric': ['euclidean', 'manhattan']},
    scoring='roc_auc', cv=5, n_jobs=-1
)
knn_grid.fit(X_train, y_train)
print('Лучшие параметры KNN:', knn_grid.best_params_, '| CV AUC:', round(knn_grid.best_score_, 4))

# Оценка лучших моделей на тесте
best_lr  = lr_grid.best_estimator_
best_dt  = dt_grid.best_estimator_
best_knn = knn_grid.best_estimator_

auc_lr  = roc_auc_score(y_test, best_lr.predict_proba(X_test)[:, 1])
auc_dt  = roc_auc_score(y_test, best_dt.predict_proba(X_test)[:, 1])
auc_knn = roc_auc_score(y_test, best_knn.predict_proba(X_test)[:, 1])

print(f'\nROC-AUC на тесте:')
print(f'  Logistic Regression: {auc_lr:.4f}')
print(f'  Decision Tree:       {auc_dt:.4f}')
print(f'  KNN:                 {auc_knn:.4f}')

# ROC-кривые
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, best_lr.predict_proba(X_test)[:, 1])
fpr_dt,  tpr_dt,  _ = roc_curve(y_test, best_dt.predict_proba(X_test)[:, 1])
fpr_knn, tpr_knn, _ = roc_curve(y_test, best_knn.predict_proba(X_test)[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr,  tpr_lr,  label=f'LR  (AUC={auc_lr:.3f})')
plt.plot(fpr_dt,  tpr_dt,  label=f'DT  (AUC={auc_dt:.3f})')
plt.plot(fpr_knn, tpr_knn, label=f'KNN (AUC={auc_knn:.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC-кривые лучших моделей')
plt.legend()
plt.show()